### base MUPRS (impactos da enchente de 2024 por município do RS)

In [1]:
import csv
import unicodedata

with open('../dados/originais/MUPRS_2024-09-03.csv', 'r', encoding='utf-8-sig') as f:
    linhas_brutas = list(csv.DictReader(f, delimiter=';'))

print(len(linhas_brutas), "linhas")
print(linhas_brutas[0])

497 linhas
{'CODIGO IBGE': '430003', 'MUNICÍPIO': 'Aceguá', 'Situação do Município': 'Emergência', 'COREDE': 'Campanha', 'Censo | População Atingida': '1', 'Censo | População Total': '4170', 'Censo | Domicílios Particulares Atingidos': '1', 'Censo | Total de Domicílios Particulares': '2123', 'Censo | Endereços Atingidos': '2', 'Censo | Total de Endereços': '3075', 'CADUnico | Famílias Atingidas': '', 'CADUnico | Total Famílias': '884', 'CADUnico | Pessoas Atingidas': '', 'CADUnico | Total Pessoas': '2142', 'CNPJ | CNPJs Atingidos': '', 'CNPJ | Total de CNPJs': '555', 'Equipamentos | Equipamentos Atingidos': '', 'Equipamentos | Total de Equipamentos': '11', 'Malha Rodoviária | Km Atingidos': '20', 'Malha Rodoviária | Total de KM': '654', 'Volta por Cima | Nº de Auxílios': '1', 'Volta por Cima | Valor Total (R$)': 'R$ 2.500,00', 'SOS RS | Nº de Auxílios': '', 'SOS RS | Valor Total (R$)': ''}


### 1. Estrutura: colunas, duplicatas e códigos

In [2]:
codigos = [l['CODIGO IBGE'] for l in linhas_brutas]
nomes = [l['MUNICÍPIO'] for l in linhas_brutas]
linhas_como_tupla = [tuple(l.values()) for l in linhas_brutas]

print("Colunas:", len(linhas_brutas[0]))
print("Códigos IBGE repetidos:", len(codigos) - len(set(codigos)))
print("Nomes de município repetidos:", len(nomes) - len(set(nomes)))
print("Linhas totalmente repetidas:", len(linhas_como_tupla) - len(set(linhas_como_tupla)))
print("Tamanho dos códigos IBGE:", set(len(c) for c in codigos))
print("Nomes com espaços sobrando:", sum(1 for n in nomes if n != n.strip()))

situacoes = {}
for l in linhas_brutas:
    s = l['Situação do Município']
    situacoes[s] = situacoes.get(s, 0) + 1
print(situacoes)

Colunas: 24
Códigos IBGE repetidos: 0
Nomes de município repetidos: 0
Linhas totalmente repetidas: 0
Tamanho dos códigos IBGE: {6}
Nomes com espaços sobrando: 0
{'Emergência': 357, 'Não Homologado': 45, 'Calamidade': 95}


### 2. Nomes das colunas

In [3]:
novos_nomes = {
    'CODIGO IBGE': 'codigo_ibge',
    'MUNICÍPIO': 'municipio',
    'Situação do Município': 'situacao',
    'COREDE': 'corede',
    'Censo | População Atingida': 'pop_atingida',
    'Censo | População Total': 'pop_total',
    'Censo | Domicílios Particulares Atingidos': 'domicilios_atingidos',
    'Censo | Total de Domicílios Particulares': 'domicilios_total',
    'Censo | Endereços Atingidos': 'enderecos_atingidos',
    'Censo | Total de Endereços': 'enderecos_total',
    'CADUnico | Famílias Atingidas': 'cad_familias_atingidas',
    'CADUnico | Total Famílias': 'cad_familias_total',
    'CADUnico | Pessoas Atingidas': 'cad_pessoas_atingidas',
    'CADUnico | Total Pessoas': 'cad_pessoas_total',
    'CNPJ | CNPJs Atingidos': 'cnpjs_atingidos',
    'CNPJ | Total de CNPJs': 'cnpjs_total',
    'Equipamentos | Equipamentos Atingidos': 'equipamentos_atingidos',
    'Equipamentos | Total de Equipamentos': 'equipamentos_total',
    'Malha Rodoviária | Km Atingidos': 'km_atingidos',
    'Malha Rodoviária | Total de KM': 'km_total',
    'Volta por Cima | Nº de Auxílios': 'vpc_auxilios',
    'Volta por Cima | Valor Total (R$)': 'vpc_valor_rs',
    'SOS RS | Nº de Auxílios': 'sosrs_auxilios',
    'SOS RS | Valor Total (R$)': 'sosrs_valor_rs',
}
assert set(novos_nomes) == set(linhas_brutas[0].keys())

linhas = []
for l in linhas_brutas:
    nova = {}
    for nome_antigo, valor in l.items():
        nova[novos_nomes[nome_antigo]] = valor
    linhas.append(nova)

print(list(linhas[0].keys()))

['codigo_ibge', 'municipio', 'situacao', 'corede', 'pop_atingida', 'pop_total', 'domicilios_atingidos', 'domicilios_total', 'enderecos_atingidos', 'enderecos_total', 'cad_familias_atingidas', 'cad_familias_total', 'cad_pessoas_atingidas', 'cad_pessoas_total', 'cnpjs_atingidos', 'cnpjs_total', 'equipamentos_atingidos', 'equipamentos_total', 'km_atingidos', 'km_total', 'vpc_auxilios', 'vpc_valor_rs', 'sosrs_auxilios', 'sosrs_valor_rs']


### 3. Valores em dinheiro gravados como texto

In [4]:
for l in linhas[:20]:
    if l['vpc_valor_rs'] != '':
        print(l['municipio'], '|', l['vpc_valor_rs'], '|', l['sosrs_valor_rs'])

def dinheiro_para_numero(texto):
    if texto == '':
        return None
    texto = texto.replace('R$', '').replace('.', '').replace(',', '.')
    return float(texto.strip())

for l in linhas:
    l['vpc_valor_rs'] = dinheiro_para_numero(l['vpc_valor_rs'])
    l['sosrs_valor_rs'] = dinheiro_para_numero(l['sosrs_valor_rs'])

Aceguá | R$ 2.500,00 | 
Agudo | R$ 550.000,00 | R$ 138.000,00
Ajuricaba | R$ 7.500,00 | 
Alecrim | R$ 2.500,00 | 
Alegrete | R$ 797.500,00 | 
Alpestre | R$ 2.500,00 | 
Alto Feliz | R$ 2.500,00 | 
Alvorada | R$ 5.535.000,00 | R$ 1.728.000,00
Ametista do Sul | R$ 7.500,00 | 
Anta Gorda | R$ 35.000,00 | 
Antônio Prado | R$ 27.500,00 | 
Arambaré | R$ 435.000,00 | R$ 194.000,00
Araricá | R$ 22.500,00 | 
Aratiba | R$ 7.500,00 | 


### 4. Demais colunas numéricas e células vazias

In [5]:
colunas_texto = ['codigo_ibge', 'municipio', 'situacao', 'corede']
colunas_dinheiro = ['vpc_valor_rs', 'sosrs_valor_rs']
colunas_numericas = [c for c in linhas[0] if c not in colunas_texto]
colunas_total = [c for c in colunas_numericas if c.endswith('_total')]
colunas_impacto = [c for c in colunas_numericas if c not in colunas_total]

for l in linhas:
    for c in colunas_numericas:
        if c in colunas_dinheiro:
            continue
        if l[c] == '':
            l[c] = None
        else:
            l[c] = int(l[c])

print(f"{'coluna':24s} {'vazios':>7s} {'zeros':>7s}")
for c in colunas_numericas:
    vazios = sum(1 for l in linhas if l[c] is None)
    zeros = sum(1 for l in linhas if l[c] == 0)
    print(f"{c:24s} {vazios:7d} {zeros:7d}")

coluna                    vazios   zeros
pop_atingida                 136       1
pop_total                      0       0
domicilios_atingidos         136       0
domicilios_total               0       0
enderecos_atingidos          129       0
enderecos_total                0       0
cad_familias_atingidas       204       0
cad_familias_total             0       0
cad_pessoas_atingidas        204       0
cad_pessoas_total              0       0
cnpjs_atingidos              293       0
cnpjs_total                    0       0
equipamentos_atingidos       389       0
equipamentos_total             0       0
km_atingidos                  52      90
km_total                       0       0
vpc_auxilios                 232       0
vpc_valor_rs                 232       0
sosrs_auxilios               408       0
sosrs_valor_rs               408       0


In [6]:
for n, v, valor_unitario in [('vpc_auxilios', 'vpc_valor_rs', 2500), ('sosrs_auxilios', 'sosrs_valor_rs', 2000)]:
    vazio_diferente = 0
    valor_errado = 0
    for l in linhas:
        if (l[n] is None) != (l[v] is None):
            vazio_diferente += 1
        elif l[n] is not None and l[v] != l[n] * valor_unitario:
            valor_errado += 1
    print(n, "| vazios diferentes entre número e valor:", vazio_diferente,
          f"| valor diferente de número x R$ {valor_unitario}:", valor_errado)

vpc_auxilios | vazios diferentes entre número e valor: 0 | valor diferente de número x R$ 2500: 0
sosrs_auxilios | vazios diferentes entre número e valor: 0 | valor diferente de número x R$ 2000: 0


In [7]:
sem_nenhum_impacto = 0
for l in linhas:
    if all(l[c] is None for c in colunas_impacto if c not in colunas_dinheiro):
        sem_nenhum_impacto += 1

    l['km_atingidos_era_vazio'] = l['km_atingidos'] is None
    for c in colunas_impacto:
        if l[c] is None:
            l[c] = 0.0 if c in colunas_dinheiro else 0

print("Municípios sem nenhum impacto registrado:", sem_nenhum_impacto)
print("Vazios restantes:", sum(1 for l in linhas for c in colunas_numericas if l[c] is None))

Municípios sem nenhum impacto registrado: 29
Vazios restantes: 0


### 5. Coerência entre colunas

In [8]:
pares = [
    ('pop_atingida', 'pop_total'),
    ('domicilios_atingidos', 'domicilios_total'),
    ('enderecos_atingidos', 'enderecos_total'),
    ('cad_familias_atingidas', 'cad_familias_total'),
    ('cad_pessoas_atingidas', 'cad_pessoas_total'),
    ('cnpjs_atingidos', 'cnpjs_total'),
    ('equipamentos_atingidos', 'equipamentos_total'),
    ('km_atingidos', 'km_total'),
]
for atingido, total in pares:
    erros = sum(1 for l in linhas if l[atingido] > l[total])
    print(f"{atingido:24s} > {total:20s}: {erros}")

print()
print("Pessoas no CadÚnico < famílias no CadÚnico:",
      sum(1 for l in linhas if l['cad_pessoas_atingidas'] < l['cad_familias_atingidas']))
print("Domicílios atingidos > endereços atingidos:",
      sum(1 for l in linhas if l['domicilios_atingidos'] > l['enderecos_atingidos']))

pop_atingida             > pop_total           : 0
domicilios_atingidos     > domicilios_total    : 0
enderecos_atingidos      > enderecos_total     : 0
cad_familias_atingidas   > cad_familias_total  : 0
cad_pessoas_atingidas    > cad_pessoas_total   : 0
cnpjs_atingidos          > cnpjs_total         : 0
equipamentos_atingidos   > equipamentos_total  : 0
km_atingidos             > km_total            : 0

Pessoas no CadÚnico < famílias no CadÚnico: 0
Domicílios atingidos > endereços atingidos: 0


### 6. Menos pessoas atingidas do que domicílios atingidos

In [9]:
media_estado = sum(l['pop_total'] for l in linhas) / sum(l['domicilios_total'] for l in linhas)
print(f"Média do RS: {media_estado:.2f} pessoas por domicílio")
print()

suspeitos = []
for l in linhas:
    if l['domicilios_atingidos'] > 0:
        razao = l['pop_atingida'] / l['domicilios_atingidos']
        if razao < 1:
            suspeitos.append((razao, l['municipio'], l['corede'], l['pop_atingida'], l['domicilios_atingidos']))

suspeitos.sort()
for razao, municipio, corede, pessoas, domicilios in suspeitos:
    print(f"{municipio:25s} ({corede}): {pessoas} pessoas / {domicilios} domicílios = {razao:.2f}")
print(len(suspeitos), "municípios")

Média do RS: 2.05 pessoas por domicílio

Santana do Livramento     (Fronteira Oeste): 0 pessoas / 1 domicílios = 0.00
Alegria                   (Fronteira Noroeste): 11 pessoas / 89 domicílios = 0.12
São Vicente do Sul        (Vale do Jaguari): 55 pessoas / 244 domicílios = 0.23
Doutor Maurício Cardoso   (Fronteira Noroeste): 97 pessoas / 349 domicílios = 0.28
Bagé                      (Campanha): 1 pessoas / 2 domicílios = 0.50
Vale Verde                (Vale do Rio Pardo): 225 pessoas / 448 domicílios = 0.50
Riozinho                  (Paranhana Encosta da Serra): 6 pessoas / 8 domicílios = 0.75
Arambaré                  (Centro-Sul): 2134 pessoas / 2666 domicílios = 0.80
Tavares                   (Sul): 6 pessoas / 7 domicílios = 0.86
São Francisco de Assis    (Vale do Jaguari): 27 pessoas / 29 domicílios = 0.93
10 municípios


### 7. Endereços atingidos sem nenhuma pessoa atingida

In [10]:
for l in linhas:
    if l['pop_atingida'] == 0 and l['enderecos_atingidos'] > 0:
        print(f"{l['municipio']:25s} pessoas: {l['pop_atingida']} | domicílios: {l['domicilios_atingidos']}"
              f" | endereços: {l['enderecos_atingidos']}")

Caraá                     pessoas: 0 | domicílios: 0 | endereços: 2
Novo Cabrais              pessoas: 0 | domicílios: 0 | endereços: 1
Rolador                   pessoas: 0 | domicílios: 0 | endereços: 1
Sagrada Família           pessoas: 0 | domicílios: 0 | endereços: 1
Saldanha Marinho          pessoas: 0 | domicílios: 0 | endereços: 1
Santana do Livramento     pessoas: 0 | domicílios: 1 | endereços: 2
Tuparendi                 pessoas: 0 | domicílios: 0 | endereços: 3
Vila Nova do Sul          pessoas: 0 | domicílios: 0 | endereços: 1


**Onde aparece.** Nos municípios acima, que têm de 1 a 3 endereços atingidos e nenhuma pessoa.

**Por que pode não ser erro:** "endereços" incluem comércio, galpões e prédios públicos, onde ninguém mora.

**Decisão:** manter e marcar com `flag`. São valores muito pequenos.

### 8. Municípios "Não Homologados" com impacto

In [11]:
nao_homologados = [l for l in linhas if l['situacao'] == 'Não Homologado']
print("Municípios não homologados:", len(nao_homologados))
print("... com população atingida:", sum(1 for l in nao_homologados if l['pop_atingida'] > 0))
print("... com auxílio Volta por Cima pago:", sum(1 for l in nao_homologados if l['vpc_auxilios'] > 0))

for l in nao_homologados:
    if l['vpc_auxilios'] > 0:
        print(l['municipio'], '| pessoas atingidas:', l['pop_atingida'], '| auxílios:', l['vpc_auxilios'],
              '| valor: R$', l['vpc_valor_rs'])

Municípios não homologados: 45
... com população atingida: 19
... com auxílio Volta por Cima pago: 1
Capão da Canoa | pessoas atingidas: 26 | auxílios: 1 | valor: R$ 2500.0


### 9. Colunas novas para análise e para cruzar com outras bases

In [12]:
def normalizar_nome(nome):
    sem_acento = unicodedata.normalize('NFKD', nome).encode('ascii', 'ignore').decode()
    return sem_acento.replace("'", '').upper().strip()

def percentual(parte, total):
    return round(parte / total * 100, 2)

contagem_flags = {}
for l in linhas:
    l['municipio_chave'] = normalizar_nome(l['municipio'])
    l['pct_pop_atingida'] = percentual(l['pop_atingida'], l['pop_total'])
    l['pct_domicilios_atingidos'] = percentual(l['domicilios_atingidos'], l['domicilios_total'])
    l['pct_cad_pessoas_atingidas'] = percentual(l['cad_pessoas_atingidas'], l['cad_pessoas_total'])
    l['pct_cnpjs_atingidos'] = percentual(l['cnpjs_atingidos'], l['cnpjs_total'])

    motivos = []
    if l['domicilios_atingidos'] > 0 and l['pop_atingida'] / l['domicilios_atingidos'] < 1:
        motivos.append('menos pessoas atingidas que domicílios atingidos')
    if l['pop_atingida'] == 0 and l['enderecos_atingidos'] > 0:
        motivos.append('endereços atingidos sem pessoas atingidas')
    if l['situacao'] == 'Não Homologado' and (l['pop_atingida'] > 0 or l['vpc_auxilios'] > 0):
        motivos.append('não homologado com impacto registrado')
    l['flag'] = '; '.join(motivos)

    if l['flag'] != '':
        contagem_flags[l['flag']] = contagem_flags.get(l['flag'], 0) + 1

    l['km_atingidos_era_vazio'] = l.pop('km_atingidos_era_vazio')

print(contagem_flags)
print()
for l in linhas:
    if l['municipio'] == 'Porto Alegre':
        for chave, valor in l.items():
            print(f"{chave:26s} {valor}")

{'não homologado com impacto registrado': 19, 'menos pessoas atingidas que domicílios atingidos': 9, 'endereços atingidos sem pessoas atingidas': 7, 'menos pessoas atingidas que domicílios atingidos; endereços atingidos sem pessoas atingidas': 1}

codigo_ibge                431490
municipio                  Porto Alegre
situacao                   Calamidade
corede                     Metropolitano Delta do Jacuí
pop_atingida               125274
pop_total                  1332845
domicilios_atingidos       69631
domicilios_total           686846
enderecos_atingidos        84836
enderecos_total            757512
cad_familias_atingidas     30366
cad_familias_total         197281
cad_pessoas_atingidas      56438
cad_pessoas_total          405369
cnpjs_atingidos            28422
cnpjs_total                190017
equipamentos_atingidos     214
equipamentos_total         1383
km_atingidos               566
km_total                   3072
vpc_auxilios               22855
vpc_valor_rs         

### 10. Verificações finais

In [13]:
assert len(linhas) == 497
assert len(set(l['codigo_ibge'] for l in linhas)) == 497
assert len(set(l['municipio_chave'] for l in linhas)) == 497
assert all(l[c] is not None for l in linhas for c in colunas_numericas)
for atingido, total in pares:
    assert all(l[atingido] <= l[total] for l in linhas), atingido
assert all(l['vpc_valor_rs'] == l['vpc_auxilios'] * 2500 for l in linhas)
assert all(l['sosrs_valor_rs'] == l['sosrs_auxilios'] * 2000 for l in linhas)

print("Todas as verificações passaram.")
pop_rs = sum(l['pop_atingida'] for l in linhas)
print(f"População atingida no RS: {pop_rs:,}".replace(',', '.'))

Todas as verificações passaram.
População atingida no RS: 970.790


### 11. Salvar

In [14]:
with open('../dados/processados/muprs_limpo.csv', 'w', encoding='utf-8', newline='') as f:
    escritor = csv.DictWriter(f, fieldnames=list(linhas[0].keys()))
    escritor.writeheader()
    escritor.writerows(linhas)

colunas_inconsistencias = ['codigo_ibge', 'municipio', 'situacao', 'pop_atingida',
                           'domicilios_atingidos', 'enderecos_atingidos', 'vpc_auxilios', 'flag']
inconsistencias = []
for l in linhas:
    if l['flag'] != '':
        inconsistencias.append({c: l[c] for c in colunas_inconsistencias})

with open('../dados/processados/muprs_inconsistencias.csv', 'w', encoding='utf-8', newline='') as f:
    escritor = csv.DictWriter(f, fieldnames=colunas_inconsistencias)
    escritor.writeheader()
    escritor.writerows(inconsistencias)

print("Salvos: muprs_limpo.csv", len(linhas), "linhas | muprs_inconsistencias.csv", len(inconsistencias), "linhas")

Salvos: muprs_limpo.csv 497 linhas | muprs_inconsistencias.csv 36 linhas
